In [ ]:
import torch
import umap
import matplotlib.pyplot as plt
from matplotlib.offsetbox import OffsetImage, AnnotationBbox

import numpy as np
from PIL import Image
from transformers import CLIPProcessor, CLIPModel
from datasets import load_dataset

In [ ]:
model = CLIPModel.from_pretrained('openai/clip-vit-base-patch32')
processor = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')

In [ ]:
def get_clip_embeddings(images, texts):
    inputs = processor(text = texts, images = images, return_tensors= 'pt', padding=True, truncation=True)
    with torch.no_grad():
        # print(inputs)
        #pixel_values [B, 3, 224, 224]
        vision_out = model.vision_model(pixel_values = inputs['pixel_values'])
        # print(vision_out)
        #image_out [B, 512]
        image_embeds = model.visual_projection(vision_out.pooler_output)
        text_out = model.text_model(input_ids = inputs['input_ids'], attention_mask = inputs['attention_mask'])
        text_embeds = model.text_projection(text_out.pooler_output)
    return image_embeds.cpu().numpy(), text_embeds.cpu().numpy()

In [ ]:
dataset = load_dataset('clip-benchmark/wds_flickr8k')
train_dataset = dataset['test'] # 학습용이 아니라 일반화가 잘 되는지 확인만
train_dataset

# Dataset({
#     features: ['__key__', '__url__', 'jpg', 'txt'],
#     num_rows: 1000
# })

In [ ]:
sample_size = 100
subset = train_dataset.select(range(sample_size))

images = list(subset['jpg'])
captions = list(subset['txt'])


In [ ]:
image_embeds, text_embeds = get_clip_embeddings(images, captions)

In [ ]:
all_embeds = np.concatenate([image_embeds,text_embeds], axis=0)

In [ ]:
umap_model = umap.UMAP(n_neighbors=15, min_dist=0.1, metric = 'cosine')
reduce_embeds = umap_model.fit_transform(all_embeds)

In [ ]:
image_coords = reduce_embeds[:sample_size]
text_coords = reduce_embeds[sample_size:]

In [ ]:
fig, ax = plt.subplots(figsize=(30,30))
ax.scatter(text_coords[:,0], text_coords[:, 1], color = 'red', label='Captions',alpha=0.5)

def plot_with_images(ax, coords, images, captions, is_text=False):
    for i, (x,y)in enumerate(coords):
        if is_text:
            ax.text(x,y, str(captions[i])[:30], fontsize = 8, color='red', ha='right')
        else:
            img = images[i].resize((64,64))
            imagebox = OffsetImage(img, zoom=1.0)
            ab = AnnotationBbox(imagebox, (x,y), frameon=False)
            ax.add_artist(ab)

plot_with_images(ax, image_coords, images, captions, is_text=False)

all_coords=np.vstack([image_coords, text_coords])
x_min, y_min = all_coords.min(axis = 0)
x_max, y_max = all_coords.max(axis = 0)
pad_x = (x_max - x_min) * 0.05
pad_y = (y_max - y_min) * 0.05
ax.set_xlim(x_min - pad_x, x_max + pad_x)
ax.set_ylim(y_min - pad_y, y_max + pad_y)

ax.legend()
plt.tight_layout()
plt.savefig('test.png', dpi=200)
plt.show()
